In [ ]:
import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)
from src.api.APIs import (
    getEstadoCirculacionesTecnicas,
    getPlanificacionCirculacionesTecnicas)
from src.utils.util import rellenarId
from src.utils.timeformat import (
    time2localtime,
    time2iso
)
from datetime import datetime

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\BI\Sitra\Circulación_sitra_2025-06-22.csv")
sitra = pd.read_csv(fname, encoding='utf-8')

In [ ]:
sitra.drop(columns={"Fecha Origen Tren (DD/MM/YY) F","Hora Teorica Llegada_Comercial (HH:MI)","Hora Teorica Salida_Comercial (HH:MI)","Fecha Teorica Llegada_Comercial (DD/MM/YYYY) F","Cod EEFF","Desc Tipo_Tren","Secuencia_Mallas","Cod Producto","Fecha Teorica Salida_Comercial (DD/MM/YYYY) F"},inplace=True)

In [ ]:
renamed_columns = {"Cod NUM_Tren":"NTécnico", "Fecha Origen Tren (YYYYMMDD) N":"FechaOrigen","Desc EEFF":"EEFF",
                   "Desc Producto":"Producto","Fecha Teorica Llegada_Comercial (YYYYMMDD) N":"FechaTeoricaLLegada",
                   "Desc  PR_Salida_Comercial":"Origen","Fecha Teorica Salida_Comercial (YYYYMMDD) N": "FechaTeoricaSalida",
                   "Cod PR_Salida_Comercial":"CódigoOrigen","Cod PR_Llegada_Comercial":"CódigoDestino","Desc  PR_Llegada_Comercial":"Destino"}

In [ ]:
sitra=sitra.rename(columns=renamed_columns)

In [ ]:
sitra

In [ ]:
sitra = sitra[["FechaOrigen","NTécnico","Producto","Origen","CódigoOrigen","FechaTeoricaSalida","Destino","CódigoDestino","FechaTeoricaLLegada"]]

In [ ]:
sitra.sort_values(by=["NTécnico"])

In [ ]:
sitra["NTécnico"] = sitra["NTécnico"].apply(rellenarId)

In [ ]:
sitra

In [ ]:
gct = getPlanificacionCirculacionesTecnicas ("2025-06-22")

In [ ]:
gct

In [ ]:
gct[gct["NTécnico"] == "52422"]

<h1>Total Trenes </h1>

In [ ]:
nt_sitra = sitra.groupby(["NTécnico"]).size().reset_index(name="conteo")
dos_tramos_sitra = nt_sitra[nt_sitra["conteo"] > 1]

In [ ]:
sitra = sitra.drop_duplicates(subset=["NTécnico"])

In [ ]:
duplicado_sitra = sitra[sitra["NTécnico"].duplicated()]

In [ ]:
duplicado_sitra

In [ ]:
gct.sort_values(by=["NTécnico"], inplace=True)

In [ ]:
duplicado_gct = gct[gct["NTécnico"].duplicated()]
duplicado_gct


In [ ]:
nt_gct = gct.groupby(["NTécnico"]).size().reset_index(name="conteo")

In [ ]:
dos_tramos = nt_gct[nt_gct["conteo"] > 1]

In [ ]:
dos_tramos.reset_index(drop=True, inplace=True)

In [ ]:
dos_tramos


In [ ]:
gct = gct.drop_duplicates(subset=["NTécnico"])

In [ ]:
gct = gct[gct["esVirtual"] == False]


In [ ]:
gct[gct["esVirtual"] == True]

In [ ]:
total_sitra = sitra["NTécnico"].count()

In [ ]:
total_gct = gct["NTécnico"].count()

In [ ]:
total_sitra

In [ ]:
total_gct

In [ ]:
gct[gct["NTécnico"] == "90074"]

<H2> NO GCT SI SITRA </H2>

In [ ]:
df_compare = pd.merge(
    sitra,
    gct[["NTécnico","Fecha","Empresa","Operador","Tipo","esComercial","esEspecial","esVirtual"]],
    on="NTécnico",
    how="outer",
    indicator=True
)

In [ ]:
df_compare

In [ ]:
solo_sitra = df_compare[df_compare["_merge"] == "left_only"].copy()

In [ ]:
solo_gct = df_compare[df_compare["_merge"] == "right_only"].copy()

In [ ]:
solo_sitra.reset_index(drop=True, inplace= True)

In [ ]:
solo_gct.reset_index (drop = True, inplace= True)

In [ ]:
solo_sitra.dropna(axis=1, how='all', inplace=True)

In [ ]:
solo_sitra.drop(columns=["_merge"], inplace=True)
solo_gct.drop(columns=["_merge"], inplace=True)

In [ ]:
solo_sitra

In [ ]:
solo_gct

In [ ]:
solo_gct.dropna(axis=1,inplace=True)

In [ ]:
solo_gct

In [ ]:
total_No_MSE = solo_sitra.count()
total_NO_SITRa = solo_gct.count()

In [ ]:
total = {"totales_Sitra":total_sitra, "totales_MSE": total_gct, to}

In [ ]:
totales = pd.DataFrame([total])

In [ ]:
totales

In [ ]:
solo_sitra["NTécnico"] = solo_sitra[["NTécnico"]].astype(str)
solo_gct ["NTécnico"] = solo_gct[["NTécnico"]].astype(str)
data = {"total_circulación": totales, "Circulación_No_MSE": solo_sitra, "Circulación_No_Sitra":solo_gct}

In [ ]:
solo_sitra

In [ ]:
today_str = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\comparacion") / f"{today_str}_circulacion_MSE_SITRA.xlsx"

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

guardarExcelMulti(data,fname)

In [ ]:
solo_gct[solo_gct["NTécnico"] =="90074"]